# 🎬 AI Video Frame Interpolation (Auto-Sync)
এই নোটবুকটি আপনার Veo 3 থেকে জেনারেট করা ভিডিও এবং Whisper-এর অডিওর গ্যাপ পূরণ করার জন্য তৈরি করা হয়েছে। এটি Optical Flow ব্যবহার করে ভিডিওর মাঝখানে নতুন ফ্রেম (Intermediate frames) তৈরি করে, যাতে ভিডিও স্লো-মোশন করলেও কোনো ফ্রেম ড্রপ না হয় বা ভিডিও না কাঁপে।

### Step 1: Install FFmpeg (Kaggle-এর GPU সার্ভারে)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
print("\n✅ FFmpeg successfully installed on Kaggle!")

### Step 2: Upload Test Video
Kaggle-এর ডানদিকের প্যানেল থেকে আপনার `final_combined_video.mp4` ভিডিওটি আপলোড করুন অথবা নিচের কোডে সঠিক পাথ দিন।

In [ ]:
import os
import subprocess

# Input and Output Paths
INPUT_VIDEO = "/kaggle/input/your-dataset/final_combined_video.mp4" # আপনার আপলোড করা ভিডিওর পাথ দিন
SNIPPET_VIDEO = "/kaggle/working/original_snippet.mp4"
OUTPUT_VIDEO = "/kaggle/working/ai_interpolated_slow.mp4"

print("Variables initialized. Ready for processing.")

### Step 3: Extract a Small Snippet & Apply AI Interpolation
এখানে আমরা ২ সেকেন্ডের একটি অংশ কেটে নেব এবং সেটিকে Motion Interpolation-এর মাধ্যমে ৪ সেকেন্ডের বানাব (২x স্লো-মোশন)।

In [ ]:
if not os.path.exists(INPUT_VIDEO):
    print(f"❌ Error: Video not found at {INPUT_VIDEO}. Please upload the video first!")
else:
    print("\n✂️ 1. Extracting a 2-second test snippet...")
    subprocess.run([
        "ffmpeg", "-y", "-ss", "00:00:10", "-i", INPUT_VIDEO, 
        "-t", "2", "-c", "copy", SNIPPET_VIDEO
    ])
    
    print("\n🧠 2. Applying AI Motion Interpolation (Generating missing frames)...")
    print("⏳ Please wait, calculating optical flow... (Very fast on Kaggle GPU)")
    
    # setpts=2.0*PTS slows it down 2x. minterpolate generates new frames.
    subprocess.run([
        "ffmpeg", "-y", "-i", SNIPPET_VIDEO,
        "-filter:v", "setpts=2.0*PTS,minterpolate='fps=30:mi_mode=mci:mc_mode=aobmc:vsbmc=1'",
        "-c:v", "libx264", "-preset", "fast", OUTPUT_VIDEO
    ])
    
    print(f"\n✅ Done! Check the '/kaggle/working/' folder for '{OUTPUT_VIDEO}'")
    print(f"Original 2s video saved as: {SNIPPET_VIDEO}")

### Step 4: Compare Results
আপনি এখন `/kaggle/working/` ফোল্ডার থেকে `original_snippet.mp4` এবং `ai_interpolated_slow.mp4` দুটি ভিডিওই ডাউনলোড করে পাশাপাশি প্লে করে দেখতে পারবেন। ইন্টারপোলেটেড ভিডিওটি স্লো হওয়া সত্ত্বেও একটুও কাঁপবে না!